Jura: ICM (多输出 GP) vs 独立 GP 基线
 
核心对比:

  - 基线: 只用 259 个 Cd 观测建单输出 GP
  - LMC : 用 259 Cd + 359 Ni + 359 Zn 建多输出 GP
  在同样的 100 个测试位置上预测 Cd, 比 RMSE / MAE / NLL
 
kernel 结构: K = K_input(x, x') * B[i, i']
  K_input : RBF, ARD (两个空间方向各一个 lengthscale)
  B       : IndexKernel, B = W @ W.T + diag(kappa), W 形状 (P, rank)
            rank 就是 LMC 里的 Q

In [1]:
import numpy as np
import torch
import gpytorch
import matplotlib.pyplot as plt
from pathlib import Path
 
ROOT = Path(r"E:\Research")
DATA = np.load(ROOT / "data" / "jura_prepared.npz", allow_pickle=True)

torch.manual_seed(0)
RANK = 2          # LMC 的 Q; 后面做敏感性分析时改这个
N_ITER = 300
LR = 0.05
LS_INIT = 0.5     # lengthscale 初值, 依据 EDA 里 5km 范围 / 采样间距 0.27km

# 张量

In [4]:
Xtr = torch.tensor(DATA["X_train"], dtype=torch.float64)
itr = torch.tensor(DATA["i_train"], dtype=torch.long)
ytr = torch.tensor(DATA["y_train"], dtype=torch.float64)
Xte = torch.tensor(DATA["X_test"],  dtype=torch.float64)
ite = torch.tensor(DATA["i_test"],  dtype=torch.long)
yte = torch.tensor(DATA["y_test"],  dtype=torch.float64)
 
mu, sd = DATA["mu"], DATA["sd"]
NAMES = list(DATA["target_names"])
P = len(NAMES)
print(f"训练观测 {len(ytr)} (各任务 {np.bincount(DATA['i_train'])}), 测试 {len(yte)}")

训练观测 977 (各任务 [259 359 359]), 测试 100


# 模型 1: LMC (Hadamard 形式, 支持缺失)

In [5]:
class LMCModel(gpytorch.models.ExactGP):
    def __init__(self, x, i, y, likelihood, num_tasks, rank):
        super().__init__((x, i), y, likelihood)
        self.mean_module = gpytorch.means.ConstantMean()
        # 注意: 这里不加 ScaleKernel。B 的对角元已经承担了 output scale,
        #       再加一个会造成参数不可辨识 (两个参数只能确定一个乘积)
        self.covar_module = gpytorch.kernels.RBFKernel(ard_num_dims=2)
        self.task_covar_module = gpytorch.kernels.IndexKernel(
            num_tasks=num_tasks, rank=rank
        )
 
    def forward(self, x, i):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)          # K_input
        covar_i = self.task_covar_module(i)     # B[i, i']
        return gpytorch.distributions.MultivariateNormal(
            mean_x, covar_x.mul(covar_i)        # 逐元素相乘 = Kronecker 的长表版本
        )
 
 
lik_lmc = gpytorch.likelihoods.GaussianLikelihood().double()
lmc = LMCModel(Xtr, itr, ytr, lik_lmc, num_tasks=P, rank=RANK).double()
lmc.covar_module.lengthscale = torch.tensor([[LS_INIT, LS_INIT]], dtype=torch.float64)
 
lmc.train(); lik_lmc.train()
opt = torch.optim.Adam(lmc.parameters(), lr=LR)
mll = gpytorch.mlls.ExactMarginalLogLikelihood(lik_lmc, lmc)
 
print("\n--- 训练 LMC ---")
for k in range(N_ITER):
    opt.zero_grad()
    loss = -mll(lmc(Xtr, itr), ytr)
    loss.backward()
    if (k + 1) % 50 == 0:
        print(f"  iter {k+1:4d}  loss={loss.item():.4f}")
    opt.step()


--- 训练 LMC ---


e:\Research\.venv\Lib\site-packages\linear_operator\utils\interpolation.py:66: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\Context.cpp:823.)
  summing_matrix = torch.sparse_coo_tensor(


  iter   50  loss=1.0347
  iter  100  loss=1.0169
  iter  150  loss=1.0145
  iter  200  loss=0.9970
  iter  250  loss=1.0013
  iter  300  loss=0.9992


# 模型2：独立GP基线

In [6]:
mask = (itr == 0)
Xcd, ycd = Xtr[mask], ytr[mask]
 
 
class SingleGP(gpytorch.models.ExactGP):
    def __init__(self, x, y, likelihood):
        super().__init__(x, y, likelihood)
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.RBFKernel(ard_num_dims=2)
        )
 
    def forward(self, x):
        return gpytorch.distributions.MultivariateNormal(
            self.mean_module(x), self.covar_module(x)
        )
 
 
lik_gp = gpytorch.likelihoods.GaussianLikelihood().double()
gp = SingleGP(Xcd, ycd, lik_gp).double()
gp.covar_module.base_kernel.lengthscale = torch.tensor(
    [[LS_INIT, LS_INIT]], dtype=torch.float64
)
 
gp.train(); lik_gp.train()
opt2 = torch.optim.Adam(gp.parameters(), lr=LR)
mll2 = gpytorch.mlls.ExactMarginalLogLikelihood(lik_gp, gp)
 
print("\n--- 训练独立 GP 基线 ---")
for k in range(N_ITER):
    opt2.zero_grad()
    loss = -mll2(gp(Xcd), ycd)
    loss.backward()
    if (k + 1) % 50 == 0:
        print(f"  iter {k+1:4d}  loss={loss.item():.4f}")
    opt2.step()


--- 训练独立 GP 基线 ---
  iter   50  loss=1.2201
  iter  100  loss=1.2163
  iter  150  loss=1.2163
  iter  200  loss=1.2163
  iter  250  loss=1.2163
  iter  300  loss=1.2163


# 评估

In [7]:
def unstd_log(y):
    """标准化尺度 -> log 浓度尺度 (task 0 = Cd)"""
    return y * sd[0] + mu[0]
 
 
def metrics(pred_dist, y_true, name):
    m = pred_dist.mean
    rmse_s = torch.sqrt(((m - y_true) ** 2).mean()).item()
    mae_s = (m - y_true).abs().mean().item()
    nll = -pred_dist.log_prob(y_true).item() / len(y_true)
 
    # 还原到原始 ppm 尺度
    #   naive : exp(后验均值)  -> 中位数意义下的还原
    #   corr  : exp(mu + s^2/2) -> 对数正态的期望
    log_m = unstd_log(m.detach().numpy())
    log_v = (pred_dist.variance.detach().numpy()) * sd[0] ** 2
    true_ppm = np.exp(unstd_log(y_true.numpy()))
    rmse_naive = np.sqrt(((np.exp(log_m) - true_ppm) ** 2).mean())
    rmse_corr = np.sqrt(((np.exp(log_m + log_v / 2) - true_ppm) ** 2).mean())
 
    print(f"\n[{name}]")
    print(f"  标准化尺度   RMSE={rmse_s:.4f}  MAE={mae_s:.4f}  NLL={nll:.4f}")
    print(f"  原始 ppm     RMSE={rmse_naive:.4f} (naive) / {rmse_corr:.4f} (lognormal 修正)")
    return dict(name=name, rmse_std=rmse_s, mae_std=mae_s, nll=nll,
                rmse_ppm=rmse_naive, mean=m.detach().numpy())
 
 
lmc.eval(); lik_lmc.eval(); gp.eval(); lik_gp.eval()
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    pred_lmc = lik_lmc(lmc(Xte, ite))
    pred_gp = lik_gp(gp(Xte))
 
print("\n" + "=" * 58)
print("测试集结果 (100 个被藏起来的 Cd 位置)")
print("=" * 58)
r_gp = metrics(pred_gp, yte, "独立 GP (只看 Cd)")
r_lmc = metrics(pred_lmc, yte, "LMC (借用 Ni/Zn)")
 
imp = (r_gp["rmse_std"] - r_lmc["rmse_std"]) / r_gp["rmse_std"] * 100
print("\n" + "-" * 58)
print(f">> LMC 相对独立 GP 的 RMSE 改善: {imp:+.2f}%")
print("-" * 58)


测试集结果 (100 个被藏起来的 Cd 位置)

[独立 GP (只看 Cd)]
  标准化尺度   RMSE=0.9715  MAE=0.7177  NLL=1.5523
  原始 ppm     RMSE=0.9088 (naive) / 0.8981 (lognormal 修正)

[LMC (借用 Ni/Zn)]
  标准化尺度   RMSE=0.7474  MAE=0.5549  NLL=1.2060
  原始 ppm     RMSE=0.6931 (naive) / 0.6877 (lognormal 修正)

----------------------------------------------------------
>> LMC 相对独立 GP 的 RMSE 改善: +23.06%
----------------------------------------------------------


# 学到的B矩阵

In [8]:
B = lmc.task_covar_module.covar_matrix.to_dense().detach().numpy()
D = np.sqrt(np.diag(B))
B_corr = B / np.outer(D, D)
 
print("\n学到的任务协方差矩阵 B:")
print(np.round(B, 4))
print("\n转成相关矩阵 (可与 EDA 的经验相关性对比):")
for r, n in enumerate(NAMES):
    print(f"  {n}: " + "  ".join(f"{B_corr[r, c]:+.3f}" for c in range(P)))
 
ls = lmc.covar_module.lengthscale.detach().numpy().ravel()
print(f"\n学到的空间 lengthscale (km): X={ls[0]:.3f}, Y={ls[1]:.3f}")
print(f"噪声方差: LMC={lik_lmc.noise.item():.4f}, 独立GP={lik_gp.noise.item():.4f}")


学到的任务协方差矩阵 B:
[[0.8909 0.6649 0.6136]
 [0.6649 0.8755 0.6459]
 [0.6136 0.6459 0.8203]]

转成相关矩阵 (可与 EDA 的经验相关性对比):
  Cd: +1.000  +0.753  +0.718
  Ni: +0.753  +1.000  +0.762
  Zn: +0.718  +0.762  +1.000

学到的空间 lengthscale (km): X=0.074, Y=0.058
噪声方差: LMC=0.0912, 独立GP=0.2862


# 有个严重问题：lengthscale 崩塌了
学到的空间 lengthscale: X=0.074 km, Y=0.058 km
EDA 测得的平均采样间距: ≈ 0.27 km

lengthscale 只有采样间距的 1/4。 这意味着 RBF kernel 在任意两个不同采样点之间的相关性 ≈ exp(-(0.27/0.07)²/2) ≈ 0.0005 —— 基本为零。

这个数字的含义，比它看起来严重

你的模型现在不是空间高斯过程，而是一个逐点回归。

它是这样工作的：测试位置 x 上，Cd 未观测，但 Ni 和 Zn 在完全相同的坐标上有观测（距离 = 0，K_input = 1）。于是模型完全靠 B[Cd,Ni]=0.66 和 B[Cd,Zn]=0.61 这两个数，从同点的 Ni/Zn 值推 Cd。周围邻居的信息一点没用上。

旁证：独立 GP 的 RMSE = 0.9715。标准化数据上，直接预测均值 0 的 RMSE 就是 1.0。也就是说独立 GP 几乎什么都没学到——它的 lengthscale 大概率也崩塌了，退化成了常数预测器（噪声方差 0.2862 相对偏高也支持这个判断）。

如果报告里不处理这件事，会被问穿：你的 23% 改善，来源不是"多输出 GP 的空间协同建模"，而是"同点线性回归"——那用普通线性回归就够了，何必上 GP。

In [11]:
ls_gp = gp.covar_module.base_kernel.lengthscale.detach().numpy().ravel()
print(f"独立GP lengthscale (km): X={ls_gp[0]:.3f}, Y={ls_gp[1]:.3f}")

独立GP lengthscale (km): X=0.096, Y=0.266


## 读数：确认了空间结构很弱，但暴露了一个新问题

| 模型 | lengthscale X | lengthscale Y | 判断 |
|---|---|---|---|
| 独立 GP | 0.096 | **0.266** | Y 方向勉强到采样间距，X 方向远低于 |
| LMC | 0.074 | 0.058 | 两个方向都远低于采样间距 |

**独立 GP 没有完全崩塌**（Y=0.266 ≈ 采样间距 0.27），但也只够看到**最近邻**，再远一格相关性就掉到 `exp(-2)≈0.14`。这解释了它 RMSE=0.9715 ——比常数预测器（1.0）好一点点，但确实只好一点点。

### 但真正值得警觉的是：两个模型学出的 lengthscale 差异巨大

同一份 Cd 数据、同一个 RBF kernel，一个学出 (0.096, 0.266)，另一个学出 (0.074, 0.058)。**Y 方向差了 4.6 倍。**

这说明**似然曲面在这个区域很平**——lengthscale 从 0.05 到 0.27 之间，边际似然几乎没差别，优化器停在哪全看初值和路径。**参数不可辨识。**

所以现在还不能下结论说"Cd 空间自相关弱"，也可能是"优化没找到真正的最优"。必须做初值扫描来区分。

---

## 系统性扫描脚本

手动改 `LS_INIT` 跑三次太原始，而且结果散在终端里没法进报告。这个脚本一次跑完所有组合并输出成表格：跑一次 20 组（5 个初值 × 独立GP + 5 × 3 个 rank），大概几分钟。

---

## 三个跑之前要知道的事

**1. 这个脚本会一次性产出报告里的两张表，别当成 debug**

- **表 A：lengthscale 初值敏感性** → 论证你的结果稳健（或者诚实报告不稳健）
- **表 B：rank（Q）的选择** → 论证你为什么选某个 Q

这两张表本来就在计划的"阶段 6 调参"里。现在做等于**提前完成了阶段 6 的一半**，不是额外工作。

**2. 判读标准我写进脚本了，但你要理解它在判什么**

- **loss 极差 < 0.01** → 所有初值实质等价，说明似然曲面确实平坦，**"Cd 空间自相关弱"是真实结论**
- **大初值 loss 明显更低** → 说明存在更好的解，之前 (0.074, 0.058) 是局部最优，**要换结果**

**注意判的是 loss（训练边际似然）不是 RMSE**。用测试 RMSE 选超参会造成测试集泄漏——这是很容易犯的错，而且报告里会被抓。

**3. 你可能会看到一个反直觉的结果：rank 越大 RMSE 反而不一定越好**

rank=3 时 B 是满秩，表达能力最强，但**参数也最多**，359 个点可能不够拟合。如果 rank=1 或 2 的 RMSE 更好，**这本身就是报告里的一个发现**："三个输出的相关结构可以用 1-2 个潜在函数充分描述"，对应地质上"共同母岩因子"的解释。

---

跑完把诊断那一段（loss 极差、最优组合、各 rank 的 RMSE）发我。这个结果决定两件事：**要不要加 lengthscale 先验**，以及**报告里 lengthscale 那段怎么写**。